# System Exploration (Security)

Parsing the data and understanding it (System attribute)

In [5]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from anomaly_detection.etl.load import load_records, load_system_df

plt.style.use('ggplot')

In [6]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/system_exploration/security"

evtx_path = project_folder / "data/raw/93_securitelog.evtx"

evtx_path

WindowsPath('c:/Users/hp/Documents/School/anomaly-detection/data/raw/93_securitelog.evtx')

In [7]:
records = load_records(evtx_path)

len(records)

31466

In [8]:
with open(project_folder / "data/processed/record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        records[0],
        indent=4
    ))

records[0]

{'#attributes': {'xmlns': 'http://schemas.microsoft.com/win/2004/08/events/event'},
 'System': {'Provider': {'Name': 'Microsoft-Windows-Security-Auditing',
   'Guid': '54849625-5478-4994-A5BA-3E3B0328C30D'},
  'EventID': 4624,
  'Version': 2,
  'Level': 0,
  'Task': 12544,
  'Opcode': 0,
  'Keywords': '0x8020000000000000',
  'TimeCreated': {'SystemTime': '2026-06-17T00:15:07.799161Z'},
  'EventRecordID': 34978816,
  'Correlation': {'ActivityID': 'F1DAB952-E970-0004-5DB9-DAF170E9DC01'},
  'Execution': {'ProcessID': 940, 'ThreadID': 5960},
  'Channel': 'Security',
  'Computer': 'ESANTE.carte.com.tn',
  'Security': None},
 'EventData': {'SubjectUserSid': 'S-1-5-18',
  'SubjectUserName': 'ESANTE$',
  'SubjectDomainName': 'CARTE',
  'SubjectLogonId': '0x3e7',
  'TargetUserSid': 'S-1-5-21-3333335972-2113272682-612058519-500',
  'TargetUserName': 'Administrateur',
  'TargetDomainName': 'ESANTE',
  'TargetLogonId': '0x77061c91',
  'LogonType': 4,
  'LogonProcessName': 'Advapi  ',
  'Authentica

In [9]:
system_df = load_system_df(evtx_path)

system_df.head()

,EventID,Version,Level,Task,Opcode,Keywords,EventRecordID,Channel,Computer,Provider_Name,Provider_Guid,TimeCreated_SystemTime,Correlation_ActivityID,Execution_ProcessID,Execution_ThreadID
0,4624,2,0,12544,0,0x8020000000000000,34978816,Security,ESANTE.carte.com.tn,Microsoft-Windows-Security-Auditing,54849625-5478-4994-A5BA-3E3B0328C30D,2026-06-17T00:15:07.799161Z,F1DAB952-E970-0004-5DB9-DAF170E9DC01,940,5960
1,4672,0,0,12548,0,0x8020000000000000,34978817,Security,ESANTE.carte.com.tn,Microsoft-Windows-Security-Auditing,54849625-5478-4994-A5BA-3E3B0328C30D,2026-06-17T00:15:07.799169Z,F1DAB952-E970-0004-5DB9-DAF170E9DC01,940,5960
2,4634,0,0,12545,0,0x8020000000000000,34978818,Security,ESANTE.carte.com.tn,Microsoft-Windows-Security-Auditing,54849625-5478-4994-A5BA-3E3B0328C30D,2026-06-17T00:15:08.470381Z,NaN,940,5960
3,4634,0,0,12545,0,0x8020000000000000,34978819,Security,ESANTE.carte.com.tn,Microsoft-Windows-Security-Auditing,54849625-5478-4994-A5BA-3E3B0328C30D,2026-06-17T00:15:55.134677Z,NaN,940,6940
4,4776,0,0,14336,0,0x8020000000000000,34978820,Security,ESANTE.carte.com.tn,Microsoft-Windows-Security-Auditing,54849625-5478-4994-A5BA-3E3B0328C30D,2026-06-17T00:16:00.615230Z,F1DAB952-E970-0001-3651-E8F170E9DC01,940,6940


In [10]:
system_df.columns

Index(['EventID', 'Version', 'Level', 'Task', 'Opcode', 'Keywords',
       'EventRecordID', 'Channel', 'Computer', 'Provider_Name',
       'Provider_Guid', 'TimeCreated_SystemTime', 'Correlation_ActivityID',
       'Execution_ProcessID', 'Execution_ThreadID'],
      dtype='str')

# EventID

In [11]:
events_df = pd.DataFrame(system_df['EventID'].value_counts())

total_events = events_df['count'].sum()

events_df['percentage'] = (events_df['count'] / total_events) * 100

events_df

,count,percentage
EventID,,
4624,6336,20.13602
4672,6335,20.132842
4634,6194,19.684739
4648,6189,19.668849
4776,6185,19.656137
4799,109,0.346406
4702,49,0.155724
5379,30,0.095341
4662,11,0.034958


In [12]:
keep_list = events_df[events_df['count'] > 10]

keep_list.index

Index([4624, 4672, 4634, 4648, 4776, 4799, 4702, 5379, 4662, 4697], dtype='Int64', name='EventID')

In [13]:
# Add the descriptions to the event IDs
event_descriptions = {
    "4624": "An account was successfully logged on",
    "4672": "Special privileges assigned to new logon",
    "4634": "An account was logged off",
    "4648": "A logon was attempted using explicit credentials",
    "4776": "The computer attempted to validate the credentials for an account",
    "4799": "A security-enabled local group membership was enumerated",
    "4702": "A scheduled task was updated",
    "5379": "Credential Manager credentials were read",
    "4662": "An operation was performed on an object",
    "4697": "A service was installed in the system",
    "4798": "A user's local group membership was enumerated",
    "4611": "A trusted logon process has been registered with the Local Security Authority",
    "5058": "Key file operation",
    "5061": "Cryptographic operation",
    "5059": "Key migration operation",
    "4699": "A scheduled task was deleted",
    "4698": "A scheduled task was created",
}

desc_series = pd.Series(event_descriptions, name='description')
desc_series.index = desc_series.index.astype(events_df.index.dtype)  # match dtype (str vs int)

events_df = events_df.join(desc_series)

events_df.to_csv(output_path / "events.txt")

events_df

,count,percentage,description
EventID,,,
4624,6336,20.13602,An account was successfully logged on
4672,6335,20.132842,Special privileges assigned to new logon
4634,6194,19.684739,An account was logged off
4648,6189,19.668849,A logon was attempted using explicit credentials
4776,6185,19.656137,The computer attempted to validate the credent...
4799,109,0.346406,A security-enabled local group membership was ...
4702,49,0.155724,A scheduled task was updated
5379,30,0.095341,Credential Manager credentials were read
4662,11,0.034958,An operation was performed on an object


In [14]:
unique_event_ids = [4611, 5058, 5061, 5059, 4699, 4698]

unique_events = [record for record in records if record['System']['EventID'] in unique_event_ids]

with open(output_path / "unique_event_records.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        unique_events,
        indent=4
    ))

unique_events

[{'#attributes': {'xmlns': 'http://schemas.microsoft.com/win/2004/08/events/event'},
  'System': {'Provider': {'Name': 'Microsoft-Windows-Security-Auditing',
    'Guid': '54849625-5478-4994-A5BA-3E3B0328C30D'},
   'EventID': 4611,
   'Version': 0,
   'Level': 0,
   'Task': 12289,
   'Opcode': 0,
   'Keywords': '0x8020000000000000',
   'TimeCreated': {'SystemTime': '2026-06-17T09:48:35.572564Z'},
   'EventRecordID': 35010093,
   'Correlation': {'ActivityID': 'F1DAB952-E970-0004-5DB9-DAF170E9DC01'},
   'Execution': {'ProcessID': 940, 'ThreadID': 6200},
   'Channel': 'Security',
   'Computer': 'ESANTE.carte.com.tn',
   'Security': None},
  'EventData': {'SubjectUserSid': 'S-1-5-18',
   'SubjectUserName': 'ESANTE$',
   'SubjectDomainName': 'CARTE',
   'SubjectLogonId': '0x3e7',
   'LogonProcessName': 'Winlogon'}},
 {'#attributes': {'xmlns': 'http://schemas.microsoft.com/win/2004/08/events/event'},
  'System': {'Provider': {'Name': 'Microsoft-Windows-Security-Auditing',
    'Guid': '5484962

# Correlation

In [15]:
correlation_activity_df = pd.DataFrame(system_df['Correlation_ActivityID'].value_counts())

correlation_activity_df

,count
Correlation_ActivityID,
F1DAB952-E970-0004-5DB9-DAF170E9DC01,19084
F1DAB952-E970-0001-3651-E8F170E9DC01,1
F1DAB952-E970-0004-F545-A1F270E9DC01,1
F1DAB952-E970-0002-B68E-E4F170E9DC01,1
F1DAB952-E970-000B-A3FA-ECF170E9DC01,1
...,...
F1DAB952-E970-0003-F748-4FF370E9DC01,1
F1DAB952-E970-000A-C5C0-07F270E9DC01,1
F1DAB952-E970-000B-2F1B-EDF170E9DC01,1


In [16]:
main_corr_id = system_df[system_df["Correlation_ActivityID"] == "F1DAB952-E970-0004-5DB9-DAF170E9DC01"]["EventID"].value_counts()
other_corr_id = system_df[system_df["Correlation_ActivityID"] != "F1DAB952-E970-0004-5DB9-DAF170E9DC01"]["EventID"].value_counts()

corr_comparison_df = pd.concat(
    [main_corr_id, other_corr_id],
    axis=1,
    keys=['MainCorrelationActivityID', 'OtherCorrelationActivityID']
)

corr_comparison_df = corr_comparison_df.fillna(0)

corr_comparison_df

,MainCorrelationActivityID,OtherCorrelationActivityID
EventID,,
4624,6336,0
4672,6335,0
4648,6189,0
4799,109,0
4702,49,0
5379,30,0
4662,11,0
4697,11,0
4798,9,0


# Execution

In [17]:
execution_df = pd.DataFrame(system_df[['Execution_ProcessID', 'Execution_ThreadID']].value_counts())

execution_df

count
Execution_ProcessID Execution_ThreadID       
940                 6940                 6938
                    6016                 6788
                    5132                  502
                    7172                  453
                    7480                  447
...                                       ...
                    6828                    4
                    2732                    4
                    8732                    4
                    3968                    4
                    9752                    4

[171 rows x 1 columns]

In [18]:
eventid_to_threadid_df = pd.crosstab(system_df["Execution_ThreadID"], system_df["EventID"])

eventid_to_threadid_df

EventID,4611,4624,4634,4648,4662,4672,4697,4698,4699,4702,4776,4798,4799,5058,5059,5061,5379
Execution_ThreadID,,,,,,,,,,,,,,,,,
132,0,65,67,65,0,65,0,0,0,1,65,0,0,0,0,0,0
972,0,49,48,48,0,49,0,0,0,0,48,0,0,0,0,0,0
1012,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0
1164,0,26,19,24,0,26,0,0,0,1,24,0,0,0,0,0,0
1176,0,7,7,6,0,7,0,0,0,0,6,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9116,0,13,12,13,0,13,0,0,0,0,13,0,0,0,0,0,0
9176,0,11,11,11,0,11,0,0,0,0,11,0,0,0,0,0,0
9196,0,17,18,17,0,17,0,0,0,0,17,0,0,0,0,0,0


# Others

In [19]:
provider_df = pd.DataFrame(system_df[['Provider_Name', 'Provider_Guid']].value_counts())

provider_df

,,count
Provider_Name,Provider_Guid,
Microsoft-Windows-Security-Auditing,54849625-5478-4994-A5BA-3E3B0328C30D,31466


In [20]:
system_df.astype(str).nunique()

EventID                      17
Version                       3
Level                         1
Task                         10
Opcode                        1
Keywords                      1
EventRecordID             31466
Channel                       1
Computer                      1
Provider_Name                 1
Provider_Guid                 1
TimeCreated_SystemTime    31465
Correlation_ActivityID     6186
Execution_ProcessID           1
Execution_ThreadID          171
dtype: int64

In [21]:
constants_df = pd.DataFrame(system_df[['Level', 'Opcode', 'Keywords', 'Channel', 'Computer', 'Execution_ProcessID']].value_counts())

constants_df.to_csv(output_path / "constants.csv")

constants_df

,,,,,,count
Level,Opcode,Keywords,Channel,Computer,Execution_ProcessID,
0,0,0x8020000000000000,Security,ESANTE.carte.com.tn,940,31466


In [22]:
eventid_to_task_df = pd.crosstab(system_df["Task"], system_df["EventID"])

eventid_to_task_df

EventID,4611,4624,4634,4648,4662,4672,4697,4698,4699,4702,4776,4798,4799,5058,5059,5061,5379
Task,,,,,,,,,,,,,,,,,
12289,3,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0
12290,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
12292,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0
12544,0,6336,0,6189,0,0,0,0,0,0,0,0,0,0,0,0,0
12545,0,0,6194,0,0,0,0,0,0,0,0,0,0,0,0,0,0
12548,0,0,0,0,0,6335,0,0,0,0,0,0,0,0,0,0,0
12804,0,0,0,0,11,0,0,1,1,49,0,0,0,0,0,0,0
13824,0,0,0,0,0,0,0,0,0,0,0,9,0,0,0,0,30
13826,0,0,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0


In [23]:
version_df = pd.DataFrame(system_df['Version'].value_counts())

version_df

,count
Version,
0,25066
2,6336
1,64


In [24]:
eventid_to_version_df = pd.crosstab(system_df["Version"], system_df["EventID"])

eventid_to_version_df

EventID,4611,4624,4634,4648,4662,4672,4697,4698,4699,4702,4776,4798,4799,5058,5059,5061,5379
Version,,,,,,,,,,,,,,,,,
0,3,0,6194,6189,11,6335,0,0,0,0,6185,9,109,0,0,1,30
1,0,0,0,0,0,0,11,1,1,49,0,0,0,1,1,0,0
2,0,6336,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
